# S1_F2_Preprocesamiento — Selección, transformación y validación
**Dataset:** 2023 Youth Risk Behavior Survey (YRBS), CDC — national (site=XX), 20.103 registros, 117 columnas originales.

**Pregunta de investigación:** ¿Qué relación existe entre los patrones de uso de tecnología y los indicadores de salud mental autopercibida y horas de sueño en los estudiantes incluidos en el dataset YRBS 2023, considerando variables sociodemográficas y de actividad física? 

**Fuente:** Centers for Disease Control and Prevention (CDC). (2024). *2023 Youth Risk Behavior Survey Data* [conjunto de datos]. https://www.cdc.gov/yrbs/data/index.html

**Objetivo general (Fase 2)**. Construir un _pipeline_ reproducible que deje el _dataset_ limpio, codificado y escalado, listo para el modelado posterior.

In [15]:
import sys
from pathlib import Path

# Agrega src/ de la raiz del proyecto al path
sys.path.append(str(Path('..') / '..' / 'src'))

import pandas as pd
import numpy as np

from procesamiento import (
    cargar_datos, seleccionar_columnas, diagnosticar_datos,
    clasificar_variables, transformar_datos, validar_datos,
    guardar_dataset, COLUMNAS_ANALISIS, COLUMNAS_SIN_DEPENDENCIA,
    ESCALAS_ORDINALES,
)

# Ruta relativa al notebook: F2/notebooks/ -> F1/data/raw/
RUTA_RAW = Path('..') / '..' / 'F1' / 'data' / 'raw' / 'XXH2023_YRBSS_data.csv'
df = cargar_datos(str(RUTA_RAW))
print('Dimensiones del archivo original:', df.shape)

Dimensiones del archivo original: (20103, 117)


## Configuración del entorno y las rutas
Se comprueba el entorno y se declaran las rutas. 

In [4]:
from pathlib import Path      # manejo de rutas independiente del sistema operativo
import sys

# sys.version trae la versión completa con fecha de compilación;
# split()[0] deja solo el número, que es lo único que hay que comprobar.
print("Python:", sys.version.split()[0])
for lib, mod in [("pandas", pd)]:
    print(f"{lib:8}:", mod.__version__)

# Estructura del proyecto. parents=True crea las carpetas intermedias;
# exist_ok=True evita el error si ya existen.
DIR_CRUDO = Path("data/raw")            # datos originales: nunca se modifican
DIR_PROCESADO = Path("data/processed")  # resultado del pipeline
DIR_DOCS = Path("docs")                 # diccionario, bitácora y metadatos
for carpeta in (DIR_CRUDO, DIR_PROCESADO, DIR_DOCS):
    carpeta.mkdir(parents=True, exist_ok=True)

ARCHIVO = DIR_CRUDO / "yrbs2023_seleccion_procesada.csv"
print("\nArchivo esperado en:", ARCHIVO)

Python: 3.14.7
pandas  : 3.0.6

Archivo esperado en: data\raw\yrbs2023_seleccion_procesada.csv


### Procedencia del conjunto de datos

| Campo | Valor |
|:--|:--|
| Título | 2023 National Youth Risk Behavior Survey (YRBS)|
| Autor institucional | Centers for Disease Control and Prevention (CDC) |
| Plataforma | Sitio web del CDC |
| Fuente oficial | https://www.cdc.gov/yrbs/data/index.html |
| Dimensiones del CSV utilizado | 20.103 filas × 117 columnas |
| Unidad de observación | Respuesta de un estudiante encuestado |

#### Referencia en APA 7 para el informe
Centers for Disease Control and Prevention. (2024). _2023 National Youth Risk Behavior Survey Data_ [Conjunto de datos]. https://www.cdc.gov/yrbs/data/index.html

## 1. Hallazgo de calidad: dtype mixto en q6orig
`q6orig` mezcla texto (p. ej. "N N") con códigos numéricos, lo que generaba un `DtypeWarning` al inferir el tipo automáticamente. Se resolvió declarando `dtype={'q6orig': 'string'}` en `cargar_datos`. No se descarta la columna: se documenta como un hallazgo de calidad real del archivo original.

In [12]:
print(df['q6orig'].dtype)
df['q6orig'].value_counts(dropna=False).head(10)

string


q6orig
504    1686
506    1684
507    1633
505    1589
503    1552
508    1540
510    1365
509    1334
502    1256
511    1111
Name: count, dtype: Int64

## 2. orig_rec: columna 100% vacía
El validador genérico señaló `orig_rec` con más de 60% de nulos. En este dataset está vacía al 100%: se elimina.

In [13]:
print('orig_rec: % nulos =', df['orig_rec'].isna().mean() * 100)
df = df.drop(columns=['orig_rec'])
print('Columnas tras eliminar orig_rec:', df.shape[1])

orig_rec: % nulos = 100.0
Columnas tras eliminar orig_rec: 116


## 3. Selección de columnas por código (desde el archivo original)
Con 117 columnas no es posible avanzar sin acotar. Se seleccionan, por código, únicamente las variables necesarias para responder la pregunta de investigación, declaradas explícitamente en `COLUMNAS_ANALISIS` (módulo `src/procesamiento.py`), con su código YRBS y su significado según el codebook oficial 2023.

In [14]:
for col_original, col_nueva in COLUMNAS_ANALISIS.items():
    print(f'{col_original:>10} -> {col_nueva}')

    record -> id_registro
    weight -> peso_muestral
   stratum -> estrato
       psu -> psu
        q1 -> edad_cod
        q2 -> sexo_cod
   raceeth -> raceeth_cod
       q84 -> salud_mental_cod
       q80 -> redes_sociales_cod
       q85 -> sueno_cod
       q76 -> actividad_fisica_cod


In [15]:
df_sel = seleccionar_columnas(df)
print('Dimensiones tras selección:', df_sel.shape)
df_sel.head()

Dimensiones tras selección: (20103, 11)


,id_registro,peso_muestral,estrato,psu,edad_cod,sexo_cod,raceeth_cod,salud_mental_cod,redes_sociales_cod,sueno_cod,actividad_fisica_cod
0,1,0.8614,103,16294,3.0,1.0,NaN,1.0,6.0,3.0,1.0
1,2,0.8920,103,16294,4.0,2.0,5.0,3.0,4.0,5.0,5.0
2,3,0.5081,103,16294,5.0,2.0,5.0,2.0,8.0,1.0,8.0
3,4,1.1759,103,16294,6.0,1.0,5.0,3.0,8.0,4.0,3.0
4,5,0.8920,103,16294,3.0,2.0,5.0,3.0,6.0,3.0,8.0


## 4. Diagnóstico de faltantes en las variables seleccionadas
Las 7 variables de análisis (`q1, q2, raceeth, q76, q80, q84, q85`) **no tienen dependencia de otra pregunta** en el Apéndice C del codebook oficial (a diferencia, por ejemplo, de Q34 que depende de haber fumado en Q33). Esto significa que, para este subconjunto, un valor nulo es **no responde genuino** (el estudiante vio la pregunta y la dejó en blanco), no "no aplica" estructural por salto de pregunta.

In [16]:
diagnostico = diagnosticar_datos(df_sel)
for k, v in diagnostico.items():
    print(f'{k}: {v}')

n_filas: 20103
n_columnas: 11
pct_nulos_por_columna: {'id_registro': np.float64(0.0), 'peso_muestral': np.float64(0.0), 'estrato': np.float64(0.0), 'psu': np.float64(0.0), 'edad_cod': np.float64(0.5), 'sexo_cod': np.float64(0.8), 'raceeth_cod': np.float64(1.8), 'salud_mental_cod': np.float64(21.9), 'redes_sociales_cod': np.float64(24.4), 'sueno_cod': np.float64(13.2), 'actividad_fisica_cod': np.float64(6.1)}
duplicados_totales: 0
tipos_de_dato: {'id_registro': 'int64', 'peso_muestral': 'float64', 'estrato': 'int64', 'psu': 'int64', 'edad_cod': 'float64', 'sexo_cod': 'float64', 'raceeth_cod': 'float64', 'salud_mental_cod': 'float64', 'redes_sociales_cod': 'float64', 'sueno_cod': 'float64', 'actividad_fisica_cod': 'float64'}


## 5. Clasificación de variables: ordinal, no numérica
El validador genérico del curso clasificaría `q80`, `q84`, etc. como "discreta" por ser códigos enteros 1-8. Esa clasificación es incorrecta para este análisis: son escalas ordinales de frecuencia (p. ej. Q84: Never < Rarely < Sometimes < Most of the time < Always), y tratarlas como numéricas asumiría que la distancia entre "rara vez" y "a veces" es igual a la distancia entre "a veces" y "siempre", lo cual no está garantizado por el diseño de la escala.

In [17]:
clasificacion = clasificar_variables(df_sel)
for col, info in clasificacion.items():
    print(f"{col:22} -> {info['rol_equipo']}")

id_registro            -> identificador
peso_muestral          -> muestral (weight)
estrato                -> muestral (stratum)
psu                    -> muestral (psu)
sexo_cod               -> nominal
raceeth_cod            -> nominal
edad_cod               -> ordinal
actividad_fisica_cod   -> ordinal
redes_sociales_cod     -> ordinal
salud_mental_cod       -> ordinal
sueno_cod              -> ordinal


## 6. Transformación: tipado categórico ordenado

In [18]:
df_t = transformar_datos(df_sel)
df_t.dtypes

id_registro                int64
peso_muestral            float64
estrato                    int64
psu                        int64
edad_cod                category
sexo_cod                 float64
raceeth_cod              float64
salud_mental_cod        category
redes_sociales_cod      category
sueno_cod               category
actividad_fisica_cod    category
dtype: object

## 7. Decisión sobre ponderación (weight, stratum, psu)
Este dataset tiene diseño muestral complejo. **Decisión del equipo para las Fases 1-2: no se pondera.** Ponderar correctamente exige software de análisis de encuestas complejas (varianza por conglomerados), fuera del alcance de esta etapa. En consecuencia, **todo resultado descriptivo de este proyecto describe la muestra de 20.103 estudiantes encuestados en 2023, y no se generaliza a la población de estudiantes de EE. UU.** Las columnas `peso_muestral`, `estrato` y `psu` se conservan en el dataset procesado para un eventual análisis ponderado en fases posteriores.

## 8. Validación

In [19]:
resultado = validar_datos(df_t)
for k, v in resultado.items():
    print(f'{k}: {v}')

columnas_obligatorias_presentes: True
id_registro_unico: True
sin_duplicados: True
edad_cod_en_rango: True
redes_sociales_cod_en_rango: True
salud_mental_cod_en_rango: True
sueno_cod_en_rango: True
actividad_fisica_cod_en_rango: True
validacion_global: True


## 9. Guardado del dataset procesado

In [20]:
# Salida del pipeline: entrada directa de la Fase 3
RUTA_SALIDA = Path('..') / 'data' / 'processed' / 'yrbs2023_seleccion_procesada.csv'
ruta_salida = guardar_dataset(df_t, str(RUTA_SALIDA))
print('Guardado en:', ruta_salida)

Guardado en: C:\Users\HUAWEI\OneDrive\Documentos\GitHub\proyecto-grupo8-mcdi500\F2\data\processed\yrbs2023_seleccion_procesada.csv


## Conclusión
El dataset se redujo de 117 a 11 columnas relevantes para la pregunta de investigación, seleccionadas por código y documentadas con su fuente en el codebook oficial. Se corrigió el hallazgo de calidad en `q6orig`, se eliminó `orig_rec` (vacía), se clasificaron las variables como ordinales (no numéricas), y se documentó la decisión de no ponderar junto con la limitación que eso implica. El dataset queda listo para el análisis de asociación en la Fase 3.